# Studying the `species` column in c14_master_v08.xlsx

This notebook starts a dedicated study of the `species` column on its own: what the raw values look like, which non-letter characters show up, and how to split/clean the values into one species per row. It picks up the species-cleaning steps first explored in `explore.ipynb` and gives them their own notebook to grow in. Matching the cleaned values against the `sead_staging` taxa tables is a separate follow-up (see `material_species_taxa_matching.ipynb` for how that was done for `material`).


In [1]:
import re

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


## Load c14_master_v08.xlsx


In [2]:
excel_path = "../data/c14_master_v08.xlsx"
df = pd.read_excel(excel_path)
print(f"{df['species'].notna().sum()} rows with species, {df['species'].nunique()} distinct values")
df['species'].head(10)


18240 rows with species, 583 distinct values


0    Lind
1     NaN
2     NaN
3     NaN
4     NaN
5     NaN
6     NaN
7     NaN
8     NaN
9     NaN
Name: species, dtype: str

### Normalize case first

Lowercasing `species` up front means case variants of the same word (e.g. `Al` / `al` / `AL`) are treated as one value throughout the rest of the notebook, instead of showing up as separate entries later.


In [3]:
df['species'] = df['species'].str.lower()


## Non-letter characters found within species values


In [4]:
non_letter_chars = (
    df['species'].dropna().astype(str)
    .apply(lambda s: sorted({ch for ch in s if not ch.isalpha()}))
)
non_letter_chars.explode().value_counts()


species
     1265
/     255
,     238
?      42
(      23
)      23
-       6
.       6
'       2
0       1
5       1
Name: count, dtype: int64

In [5]:
char_examples = {
    ch: list(df.loc[df['species'].astype(str).str.contains(re.escape(ch), na=False), 'species'].unique()[:5])
    for ch in non_letter_chars.explode().dropna().unique()
}
pd.Series(char_examples)


     [salix sp, cerealia indet, naket korn, bröd- k...
/    [får/get, vete, spelt/emmer, salix sp/asp, däg...
-     [bröd- kubbvete, spelt- emmervete, sol-fraktion]
,    [korn, naket, vete, spelt/emmer, hassel, björk...
?         [korn?, alkotte?, en?, nötkreatur?, tibast?]
0                                                  [0]
.    [hund, ospec., ev. harts, kottefjäll. tall, mä...
(    [(sol), matskorpa (sol), matskorpa (ins), pals...
)    [(sol), matskorpa (sol), matskorpa (ins), pals...
'                                                  [']
5                                             [bjö5rk]
dtype: object

## Splitting species into individual values

Values are separated by `,` or `/` (e.g. `"Vete, spelt/emmer"` lists three candidate species). Parenthetical content is left untouched, and stray digits and `?` are stripped from each split part.


In [6]:
noise_pattern = re.compile(r'[0-9?]')

def clean_text(value):
    if pd.isna(value) or value == '':
        return pd.NA
    cleaned = noise_pattern.sub('', value).strip()
    return cleaned if cleaned else pd.NA

species_parts = df['species'].str.split(r'[,/]', expand=True)
species_parts.columns = [f'species_{i + 1}' for i in range(species_parts.shape[1])]
species_parts = species_parts.apply(lambda col: col.map(clean_text))

df = df.join(species_parts)


In [7]:
species_columns = ['species'] + list(species_parts.columns)
df[species_columns].drop_duplicates().sort_values('species')


,species,species_1,species_2,species_3,species_4,species_5,species_6
14244,',',NaN,NaN,NaN,NaN,NaN
8562,(sol),(sol),NaN,NaN,NaN,NaN,NaN
3175,0,NaN,NaN,NaN,NaN,NaN,NaN
29230,aborrskinn,aborrskinn,NaN,NaN,NaN,NaN,NaN
29644,"agn, granbarr",agn,granbarr,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
18092,örtstam och rot,örtstam och rot,NaN,NaN,NaN,NaN,NaN
26137,örtstjälk,örtstjälk,NaN,NaN,NaN,NaN,NaN
21559,örtstjälkar,örtstjälkar,NaN,NaN,NaN,NaN,NaN
17500,ötkreatur,ötkreatur,NaN,NaN,NaN,NaN,NaN


## Melting into one species per row

Rows with multiple species get duplicated, one row per species, in a new `species_split` column. Rows with no valid species keep a single row with `species_split = NaN`.


In [8]:
original_row_count = len(df)
species_cols = [c for c in df.columns if c.startswith('species_')]
species_split = df[species_cols].stack().dropna().droplevel(1).rename('species_split')
df = df.drop(columns=species_cols).join(species_split).reset_index(drop=True)
print(f'{len(df)} rows after melting (was {original_row_count} before)')


30832 rows after melting (was 30301 before)


In [9]:
df[['species', 'species_split']].sort_values('species')


,species,species_split
20451,','
14455,','
8702,(sol),(sol)
3225,0,NaN
29731,aborrskinn,aborrskinn
...,...,...
30809,NaN,NaN
30810,NaN,NaN
30811,NaN,NaN
30812,NaN,NaN


## Counting how often each species_split value shows up

For each distinct `species_split` value, count rows where it appears as part of the value — e.g. `"tall"` should count a row whose original `species` was `"el/tall/nogot"`. `original_species` reconstructs the pre-melt, one-row-per-record view (via the unique `fid`) so a multi-species row isn't counted more than once just because melting duplicated it.


In [10]:
def count_contains(values, text_column):
    return (
        pd.Series({
            value: text_column.str.contains(re.escape(value), case=False, na=False, regex=True).sum()
            for value in values
        })
        .rename_axis('species_split')
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

unique_species_split = df['species_split'].dropna().unique()
original_species = df.drop_duplicates(subset='fid')['species']


In [11]:
species_split_counts_in_original = count_contains(unique_species_split, original_species)
species_split_counts_in_original


,species_split,count
3,al,6271
13,tall,3016
5,björk,2040
6,ek,1797
251,ko,1763
...,...,...
406,agn,1
409,snöre,1
408,cyperaceae,1
410,haver,1


### Also counting rows with no species value at all

Rows where `species` is missing aren't "contained in" by any token, so they're invisible in the table above. Adding an explicit `NaN` entry makes the missing rows part of the count too.


In [12]:
missing_species_count = pd.DataFrame({
    'species_split': [pd.NA],
    'count': [original_species.isna().sum()],
})
species_split_counts_in_original = pd.concat(
    [species_split_counts_in_original, missing_species_count], ignore_index=True
).sort_values('count', ascending=False)
species_split_counts_in_original


,species_split,count
411,<NA>,12061
0,al,6271
1,tall,3016
2,björk,2040
3,ek,1797
...,...,...
384,organisk massa,1
383,rosväxt,1
382,sot i keramik,1
397,brännässla,1


In [13]:
species_split_counts_in_original['count'].sum()

np.int64(43435)

## Matching species_split against the SEAD taxa tables, then GBIF

Same taxonomy hierarchy and matching approach as `material_species_taxa_matching.ipynb` (`tbl_taxa_common_names` for Swedish vernacular names, `tbl_taxa_tree_genera` / `_families` / `_orders` for Latin names, with a suffix-inference fallback for Swedish compound tree names). No tokenizing is needed here since `species_split` is already one atomic value per row.

For every `species_split` value that resolves to a SEAD taxon, its best Latin candidate (species binomial if we got one, otherwise genus) is then looked up in GBIF's taxonomic backbone (`species/match`) to cross-check it against a second, independent source, and finally GBIF's vernacular-name endpoint is used to fetch an English common name where one exists.


### Connect to sead_staging database


In [14]:
import os

from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")


### Load the SEAD taxonomy lookups


In [15]:
sv_common = pd.read_sql(
    """
    select lower(cn.common_name) as common_name_lc, cn.common_name,
           v.taxon_id, v.species, v.genus, v.family, v."order"
    from public.tbl_taxa_common_names cn
    join public.view_taxa_alphabetically v on v.taxon_id = cn.taxon_id
    where cn.language_id = 2
    """,
    engine,
)
genera = pd.read_sql('select lower(genus_name) as genus_lc, genus_name, genus_id, family_id from public.tbl_taxa_tree_genera', engine)
families = pd.read_sql('select lower(family_name) as family_lc, family_name, family_id, order_id from public.tbl_taxa_tree_families', engine)
orders_lookup = pd.read_sql('select lower(order_name) as order_lc, order_name, order_id from public.tbl_taxa_tree_orders', engine)

common_map = sv_common.drop_duplicates('common_name_lc').set_index('common_name_lc')
genus_hierarchy = (
    genera.merge(families[['family_id', 'family_name', 'order_id']], on='family_id', how='left')
          .merge(orders_lookup[['order_id', 'order_name']], on='order_id', how='left')
          .drop_duplicates('genus_lc').set_index('genus_lc')
)
family_hierarchy = (
    families.merge(orders_lookup[['order_id', 'order_name']], on='order_id', how='left')
             .drop_duplicates('family_lc').set_index('family_lc')
)
order_hierarchy = orders_lookup.drop_duplicates('order_lc').set_index('order_lc')

print(f'{len(sv_common)} Swedish common names, {len(genus_hierarchy)} genera, {len(family_hierarchy)} families, {len(order_hierarchy)} orders loaded')


4272 Swedish common names, 5212 genera, 541 families, 57 orders loaded


### Match each species_split value against the taxonomy


In [16]:
LATIN_QUALIFIER_RE = re.compile(r'^(cf\.?\s+|aff\.?\s+)|(\s+(sp\.?|spp\.?|indet\.?)\s*$)', re.I)

def strip_latin_qualifiers(value):
    prev = None
    while prev != value:
        prev = value
        value = LATIN_QUALIFIER_RE.sub('', value).strip()
    return value

def match_exact(value_lc):
    if value_lc in common_map.index:
        rec = common_map.loc[value_lc]
        return dict(match_level='species (common name)', genus=rec['genus'], family=rec['family'], order=rec['order'],
                    sead_common_name=rec['common_name'], sead_species_name=f"{rec['genus']} {rec['species']}")
    cleaned = strip_latin_qualifiers(value_lc)
    if cleaned in genus_hierarchy.index:
        rec = genus_hierarchy.loc[cleaned]
        return dict(match_level='genus (latin)', genus=rec['genus_name'], family=rec['family_name'], order=rec['order_name'],
                    sead_common_name=None, sead_species_name=None)
    if cleaned in family_hierarchy.index:
        rec = family_hierarchy.loc[cleaned]
        return dict(match_level='family (latin)', genus=None, family=rec['family_name'], order=rec['order_name'],
                    sead_common_name=None, sead_species_name=None)
    if cleaned in order_hierarchy.index:
        rec = order_hierarchy.loc[cleaned]
        return dict(match_level='order (latin)', genus=None, family=None, order=rec['order_name'],
                    sead_common_name=None, sead_species_name=None)
    return None

def infer_genus_by_suffix(value_lc):
    """Swedish tree names compound as <modifier><base>, e.g. 'klibbal' -> Alnus. Only
    accepted when every common name ending in the value converges on a single genus."""
    if len(value_lc) < 2 or not value_lc.isalpha():
        return None
    hits = sv_common[sv_common['common_name_lc'].str.endswith(value_lc)]
    if hits.empty:
        return None
    candidate_genera = hits['genus'].unique().tolist()
    if len(candidate_genera) == 1:
        rec = genus_hierarchy[genus_hierarchy['genus_name'] == candidate_genera[0]]
        return dict(match_level='genus (suffix-inferred)', genus=candidate_genera[0],
                    family=rec['family_name'].iloc[0] if len(rec) else None,
                    order=rec['order_name'].iloc[0] if len(rec) else None,
                    sead_common_name=None, sead_species_name=None)
    return dict(match_level='genus (suffix, ambiguous)', genus=None, family=None, order=None,
                sead_common_name=None, sead_species_name=None)


In [17]:
unique_species_split = df['species_split'].dropna().unique()

rows = []
for value in unique_species_split:
    result = match_exact(value) or infer_genus_by_suffix(value) or dict(
        match_level=None, genus=None, family=None, order=None, sead_common_name=None, sead_species_name=None,
    )
    result['species_split'] = value
    rows.append(result)

species_taxa_matches = pd.DataFrame(rows)[
    ['species_split', 'match_level', 'genus', 'family', 'order', 'sead_common_name', 'sead_species_name']
]
print(species_taxa_matches['match_level'].value_counts(dropna=False))
species_taxa_matches.sort_values('species_split')


match_level
NaN                          272
species (common name)         64
genus (latin)                 34
genus (suffix, ambiguous)     22
genus (suffix-inferred)       19
Name: count, dtype: int64


,species_split,match_level,genus,family,order,sead_common_name,sead_species_name
278,',NaN,NaN,NaN,NaN,NaN,NaN
211,(sol),NaN,NaN,NaN,NaN,NaN,NaN
401,aborrskinn,NaN,NaN,NaN,NaN,NaN,NaN
406,agn,NaN,NaN,NaN,NaN,NaN,NaN
3,al,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
14,örtstam,NaN,NaN,NaN,NaN,NaN,NaN
323,örtstam och rot,NaN,NaN,NaN,NaN,NaN,NaN
383,örtstjälk,NaN,NaN,NaN,NaN,NaN,NaN
353,örtstjälkar,NaN,NaN,NaN,NaN,NaN,NaN


### Cross-check the resolved candidate against GBIF

GBIF's `species/match` only works on scientific names, not vernacular ones, so the query is the SEAD-resolved Latin candidate (species binomial if we have one, otherwise genus) — never the raw Swedish `species_split` text. `kingdom=Plantae` is passed since SEAD's taxonomy is flora-only, which resolves some genus-only homonym issues (e.g. bare `Betula`/`Pinus` otherwise return no match). Rows with no SEAD candidate at all (non-taxonomic values, family/order-only matches) are skipped rather than guessed at.


In [18]:
import requests

GBIF_SESSION = requests.Session()

def gbif_match(name):
    try:
        response = GBIF_SESSION.get(
            'https://api.gbif.org/v1/species/match',
            params={'name': name, 'kingdom': 'Plantae'},
            timeout=10,
        )
        response.raise_for_status()
        data = response.json()
    except requests.RequestException:
        return None
    if data.get('matchType') in (None, 'NONE'):
        return None
    return {
        'gbif_usage_key': data.get('usageKey'),
        'gbif_canonical_name': data.get('canonicalName'),
        'gbif_rank': data.get('rank'),
        'gbif_match_type': data.get('matchType'),
        'gbif_confidence': data.get('confidence'),
    }

GBIF_FIELDS = ['gbif_usage_key', 'gbif_canonical_name', 'gbif_rank', 'gbif_match_type', 'gbif_confidence']

def gbif_lookup(candidate, cache):
    if pd.isna(candidate):
        return pd.Series({field: pd.NA for field in GBIF_FIELDS})
    result = cache.get(candidate)
    if result is None:
        return pd.Series({field: pd.NA for field in GBIF_FIELDS})
    return pd.Series(result)

species_taxa_matches['gbif_candidate'] = species_taxa_matches['sead_species_name'].fillna(species_taxa_matches['genus'])

gbif_cache = {
    candidate: gbif_match(candidate)
    for candidate in species_taxa_matches['gbif_candidate'].dropna().unique()
}
print(f'{len(gbif_cache)} distinct candidates queried against GBIF')

gbif_fields = species_taxa_matches['gbif_candidate'].apply(lambda c: gbif_lookup(c, gbif_cache))
species_taxa_matches = pd.concat([species_taxa_matches, gbif_fields], axis=1)
print(f"{species_taxa_matches['gbif_usage_key'].notna().sum()} of {len(species_taxa_matches)} species_split values matched in GBIF")
species_taxa_matches.sort_values('species_split')


101 distinct candidates queried against GBIF
117 of 411 species_split values matched in GBIF


,species_split,match_level,genus,family,order,sead_common_name,sead_species_name,gbif_candidate,gbif_usage_key,gbif_canonical_name,gbif_rank,gbif_match_type,gbif_confidence
278,',NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
211,(sol),NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
401,aborrskinn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
406,agn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
3,al,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14,örtstam,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
323,örtstam och rot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
383,örtstjälk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>
353,örtstjälkar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>


### Translate the common name to English via GBIF vernacular names

For every distinct GBIF `usageKey` found above, fetch its vernacular names and keep the first English (`eng`) one, if any exist — not every taxon has an English vernacular name recorded in GBIF, so this is populated where possible rather than guaranteed.


In [19]:
def gbif_english_name(usage_key):
    try:
        response = GBIF_SESSION.get(
            f'https://api.gbif.org/v1/species/{int(usage_key)}/vernacularNames',
            params={'limit': 50},
            timeout=10,
        )
        response.raise_for_status()
        results = response.json().get('results', [])
    except requests.RequestException:
        return pd.NA
    english_names = [r['vernacularName'] for r in results if r.get('language') in ('eng', 'en')]
    return english_names[0] if english_names else pd.NA

english_cache = {
    key: gbif_english_name(key)
    for key in species_taxa_matches['gbif_usage_key'].dropna().unique()
}
species_taxa_matches['species_split_english'] = species_taxa_matches['gbif_usage_key'].map(english_cache)
print(f"{species_taxa_matches['species_split_english'].notna().sum()} of {len(species_taxa_matches)} species_split values have an English name from GBIF")
species_taxa_matches.drop(columns='gbif_candidate').sort_values('species_split')


106 of 411 species_split values have an English name from GBIF


,species_split,match_level,genus,family,order,sead_common_name,sead_species_name,gbif_usage_key,gbif_canonical_name,gbif_rank,gbif_match_type,gbif_confidence,species_split_english
278,',NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN
211,(sol),NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN
401,aborrskinn,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN
406,agn,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN
3,al,"genus (suffix, ambiguous)",NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14,örtstam,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN
323,örtstam och rot,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN
383,örtstjälk,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN
353,örtstjälkar,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN


### Attach the results back onto df and export the lookup table


In [20]:
df = df.merge(
    species_taxa_matches.drop(columns='gbif_candidate'),
    on='species_split',
    how='left',
)
species_taxa_matches.drop(columns='gbif_candidate').to_csv(
    '../output/species_split_taxa_gbif_matches.csv', index=False
)
df.head()


,fid,socken,landskap,village_farm,raa_id,site_type,site_id,uppdragsnummer,lab_no,c14_age_bp,c14_error,d13C,pMC_value,pMC_error,c14_data_status,comment,assessed_relevant,material,species,context_id,context_type,northing_3006,easting_3006,longitude,latitude,location_precision,cal_68_min,cal_68_max,cal_95_min,cal_95_max,median_cal_year,calibration_method,calibration_status,author,publication_year,title,journal,place_of_publication,species_split,match_level,genus,family,order,sead_common_name,sead_species_name,gbif_usage_key,gbif_canonical_name,gbif_rank,gbif_match_type,gbif_confidence,species_split_english
0,1,Jörlanda,Bohuslän,Kyrkeby 3:34 m.fl.,Jörlanda 158,Boplats,L1969:5267,103367.0,Ua-49252,2841.0,33.0,25.3,NaN,NaN,ok,NaN,J,Träkol,lind,A1328,Kokgrop,6432029,312382,11.826064,57.990215,archaeological site,-930.0,-1046.0,-913.0,-1110.0,-997.0,HPD; IntCal20; rcarbon 1.5.0,calibrated,"Åberg, Joakim",2015.0,Bergsmonument och boplatser i Jörlanda,"Bohusläns museum, Rapport 2015:14",Uddevalla,lind,species (common name),Tilia,Tiliaceae,Malvales,lind,Tilia cordata,3152047,Tilia cordata,SPECIES,EXACT,99,Linden
1,2,Jörlanda,Bohuslän,Berg1:69,Jörlanda 185,Boplats,L1970:9431,NaN,T-8772,1720.0,80.0,NaN,NaN,NaN,ok,NaN,J,Träkol,NaN,A4,Stenpackning,6435710,313378,11.839973,58.023646,archaeological site,414.0,247.0,541.0,132.0,339.0,HPD; IntCal20; rcarbon 1.5.0,calibrated,"Johansson, Nils",1995.0,Tre boplatser i Spekerödsdalen,"UV Väst, Rapport 1995:1",Kungsbacka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Jörlanda,Bohuslän,Berg1:69,Jörlanda 185,Boplats,L1970:9431,NaN,T-8773,2890.0,105.0,NaN,NaN,NaN,ok,NaN,J,Träkol,NaN,A6,Stenpackning,6435710,313378,11.839973,58.023646,archaeological site,-930.0,-1214.0,-830.0,-1381.0,-1086.0,HPD; IntCal20; rcarbon 1.5.0,calibrated,"Johansson, Nils",1995.0,Tre boplatser i Spekerödsdalen,"UV Väst, Rapport 1995:1",Kungsbacka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Jörlanda,Bohuslän,Berg1:69,Jörlanda 185,Boplats,L1970:9431,NaN,T-8774,2015.0,75.0,NaN,NaN,NaN,ok,NaN,J,Träkol,NaN,A8,Härd,6435710,313378,11.839973,58.023646,archaeological site,117.0,-101.0,207.0,-333.0,-5.0,HPD; IntCal20; rcarbon 1.5.0,calibrated,"Johansson, Nils",1995.0,Tre boplatser i Spekerödsdalen,"UV Väst, Rapport 1995:1",Kungsbacka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Jörlanda,Bohuslän,Berg1:69,Jörlanda 185,Boplats,L1970:9431,NaN,T-8775,1990.0,55.0,NaN,NaN,NaN,ok,NaN,J,Träkol,NaN,A23,Grop,6435710,313378,11.839973,58.023646,archaeological site,110.0,-42.0,203.0,-144.0,29.0,HPD; IntCal20; rcarbon 1.5.0,calibrated,"Johansson, Nils",1995.0,Tre boplatser i Spekerödsdalen,"UV Väst, Rapport 1995:1",Kungsbacka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Caveat: suffix-inferred genus matches on animal/material terms are spurious

`species_split` isn't flora-only — bone/tooth/horn `material` rows carry animal terms like `ko` (cow), `svin` (pig), `häst` (horse), `get` (goat), `älg` (moose), `ren` (reindeer), `horn`, `läder` (leather) and `hår` (hair). SEAD's taxa tables are flora-only (confirmed in `material_species_taxa_matching.ipynb`), so the suffix-inference heuristic — designed for genuine Swedish compound tree names like `klibbal` → *Alnus* — occasionally finds an unrelated Swedish *plant* name that happens to end with the same letters and confidently returns the wrong genus:

- `horn` → *Ibicella* ("Yellow Unicorn-Plant")
- `hår` → *Drosera* (sundew; matches because `sileshår` ends in `hår`)
- `läder` → *Sambucus* (elder; matches because `fläder` ends in `läder`)
- `ko` → *Cypripedium* (lady's-slipper orchid)
- `ren` → *Syringa* (lilac; matches because `syren` ends in `ren`)
- `älg` → *Salix* (sallow)
- `svin` → *Parthenocissus* (Virginia creeper)
- `bröd` (bread, generic) → *Filipendula* (matches because `brudbröd` ends in `bröd`)
- `spel` (shorthand for spelt wheat) → *Amelanchier* (serviceberry)

The GBIF cross-check doesn't catch these — it only validates that the *genus name string* resolves in GBIF's backbone, not whether that genus was the right one to propose in the first place, so it ends up lending false confidence to an already-wrong guess rather than catching it. These rows are all tagged `match_level = 'genus (suffix-inferred)'`, so they can be filtered out or reviewed manually — treat that match level as lower-confidence than `'species (common name)'` or `'genus (latin)'`, both of which come from an exact lookup rather than a heuristic guess.
